# Notebook 2 — Form Calculations

Here we calculate the **form** of any team over their last N matches.

**Definition of form:** the net ELO change a team accumulated over their last N matches, using FIFA's SUM formula for each game:

$$\text{form} = \sum_{i=1}^{N} I_i \times (W_i - W_{e,i})$$

- $I$ — match importance weight (same scale as FIFA's official algorithm)
- $W$ — actual result: 1 (win), 0.5 (draw), 0 (loss)
- $W_e$ — expected result based on ELO difference between teams at the time of the match

A **positive** form score means the team consistently beat expectations. A **negative** score means they underperformed.

We walk the last N matches forward using the team's current FIFA ELO as the starting point, so the baseline is always grounded in the official present-day rating.

---
## 1. Imports & Load Data

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUT_DIR = Path('output')

results   = pd.read_csv(OUTPUT_DIR / 'results_clean.csv', parse_dates=['date'])
fifa_rank = pd.read_csv(OUTPUT_DIR / 'fifa_ranking_clean.csv')

ELO_LOOKUP = dict(zip(fifa_rank['team'], fifa_rank['fifa_elo']))

# Pre-index matches by team so get_form doesn't scan the full DataFrame each call
# Each team maps to a sorted DataFrame of their matches only
TEAM_MATCHES = {}
for team, group in pd.concat([
    results.assign(_team=results['home_team']),
    results.assign(_team=results['away_team'])
]).groupby('_team'):
    TEAM_MATCHES[team] = group.drop(columns='_team').sort_values('date').reset_index(drop=True)

print(f'Matches loaded : {len(results):,}')
print(f'Teams indexed  : {len(TEAM_MATCHES)}')

Matches loaded : 49,378
Teams indexed  : 336


---
## 2. Tournament Importance Weights

FIFA weights each match by its competitive importance (I). We use the same scale here so that form earned in a World Cup match counts more than a friendly.

Tournaments not in this map (obscure regional cups, etc.) fall back to `1.0`.

In [14]:
TOURNAMENT_WEIGHTS = {
    # Friendlies
    'Friendly':                               1.0,
    'FIFA Series':                            1.0,
    # Nations Leagues
    'UEFA Nations League':                    1.5,
    'CONCACAF Nations League':                1.5,
    'CONCACAF Nations League qualification':  1.5,
    # Qualification
    'FIFA World Cup qualification':           2.5,
    'UEFA Euro qualification':                2.5,
    'African Cup of Nations qualification':   2.5,
    'AFC Asian Cup qualification':            2.5,
    'Gold Cup qualification':                 2.5,
    'Oceania Nations Cup qualification':      2.5,
    'CONCACAF Championship qualification':    2.5,
    'Copa América qualification':             2.5,
    'AFF Championship qualification':         2.5,
    'EAFF Championship qualification':        2.5,
    'AFC Challenge Cup qualification':        2.5,
    # Confederation finals
    'UEFA Euro':                              3.5,
    'Copa América':                           3.5,
    'African Cup of Nations':                 3.5,
    'AFC Asian Cup':                          3.5,
    'Gold Cup':                               3.5,
    'CONCACAF Championship':                  3.5,
    'Oceania Nations Cup':                    3.5,
    'AFF Championship':                       3.5,
    'WAFF Championship':                      3.5,
    'SAFF Cup':                               3.5,
    'Gulf Cup':                               3.5,
    'Arab Cup':                               3.5,
    'COSAFA Cup':                             3.5,
    'CECAFA Cup':                             3.5,
    # Confederations Cup
    'Confederations Cup':                     4.0,
    # World Cup
    'FIFA World Cup':                         5.0,
}

# Tournaments we consider "competitive" (no friendlies)
FRIENDLY_TOURNAMENTS = {'Friendly', 'FIFA Series'}

results['importance'] = results['tournament'].map(TOURNAMENT_WEIGHTS).fillna(1.0)
results['is_competitive'] = ~results['tournament'].isin(FRIENDLY_TOURNAMENTS)

print('Competitive matches:', results['is_competitive'].sum())
print('Friendly matches:   ', (~results['is_competitive']).sum())

Competitive matches: 30953
Friendly matches:    18425


---
## 3. Finding a Team's Form

Form is the sum of `I × (W − Wₑ)` across the last N matches, where both teams' current FIFA ELO is used as a fixed baseline for every match. This is a simplification made to not overcomplicate or backtrack unnecessarily when the estimation is close enough.

In [19]:
FRIENDLY_TOURNAMENTS = {'Friendly', 'FIFA Series'}

def expected_result(rating_a: float, rating_b: float) -> float:
    return 1 / (10 ** (-(rating_a - rating_b) / 600) + 1)


def get_form(team: str, n: int = 10, competitive: bool = False) -> dict:
    """
    Estimate a team's form over their last N matches.

    Uses current FIFA ELO for both teams in every match as a fixed baseline —
    a deliberate simplification that avoids backtracking and is accurate enough
    for a form estimate.

    Parameters
    ----------
    team        : team name as it appears in results_clean.csv
    n           : number of matches to look back
    competitive : if True, exclude friendlies

    Returns
    -------
    dict with keys:
        form_score   – net ELO delta over last N matches (positive = above expectation)
        matches_used – actual number of matches found
        record       – (wins, draws, losses)
        matches      – DataFrame of the individual matches with per-match contribution
    """
    tm = TEAM_MATCHES.get(team)
    if tm is None:
        return {'form_score': 0.0, 'matches_used': 0, 'record': (0, 0, 0), 'matches': pd.DataFrame()}

    if competitive:
        tm = tm[~tm['tournament'].isin(FRIENDLY_TOURNAMENTS)]

    tm = tm.tail(n).copy()

    r_team = ELO_LOOKUP.get(team, 1000.0)

    contributions = []
    wins = draws = losses = 0

    for _, row in tm.iterrows():
        is_home = row['home_team'] == team
        opp     = row['away_team'] if is_home else row['home_team']
        r_opp   = ELO_LOOKUP.get(opp, 1000.0)
        I       = TOURNAMENT_WEIGHTS.get(row['tournament'], 1.0)

        we = expected_result(r_team, r_opp)

        h, a = row['home_score'], row['away_score']
        if h > a:   w = 1.0 if is_home else 0.0
        elif h < a: w = 0.0 if is_home else 1.0
        else:       w = 0.5

        contributions.append(round(I * (w - we), 4))

        if w == 1.0:   wins   += 1
        elif w == 0.5: draws  += 1
        else:          losses += 1

    tm = tm.copy()
    tm['elo_contribution'] = contributions

    return {
        'form_score':   round(sum(contributions), 4),
        'matches_used': len(tm),
        'record':       (wins, draws, losses),
        'matches':      tm[['date', 'home_team', 'away_team', 'home_score',
                             'away_score', 'tournament', 'elo_contribution']],
    }

## Sanity Tests

In [20]:
# All matches form
form = get_form('Argentina', n=10, competitive=False)
print(f"Argentina — last 10 matches")
print(f"  Form score : {form['form_score']:+.2f}")
print(f"  Record     : {form['record'][0]}W {form['record'][1]}D {form['record'][2]}L")
print()
form['matches']


Argentina — last 10 matches
  Form score : -0.86
  Record     : 8W 1D 1L



,date,home_team,away_team,home_score,away_score,tournament,elo_contribution
1058,2025-06-05,Chile,Argentina,0,1,FIFA World Cup qualification,0.4116
1059,2025-06-10,Argentina,Colombia,1,1,FIFA World Cup qualification,-0.4106
1060,2025-09-04,Argentina,Venezuela,3,0,FIFA World Cup qualification,0.4268
1061,2025-09-09,Ecuador,Argentina,1,0,FIFA World Cup qualification,-1.8593
1062,2025-10-10,Argentina,Venezuela,1,0,Friendly,0.1707
1063,2025-10-14,Puerto Rico,Argentina,0,6,Friendly,0.0367
1064,2025-11-14,Angola,Argentina,0,2,Friendly,0.0866
1065,2026-03-27,Argentina,Mauritania,2,1,Friendly,0.0628
1066,2026-03-31,Argentina,Zambia,5,0,Friendly,0.0847
1067,2026-06-06,Argentina,Honduras,2,0,Friendly,0.1292


In [21]:
# Competitive-only form
form_comp = get_form('England', n=15, competitive=True)
print(f"England — last 15 competitive matches")
print(f"  Form score : {form_comp['form_score']:+.2f}")
print(f"  Record     : {form_comp['record'][0]}W {form_comp['record'][1]}D {form_comp['record'][2]}L")
print()
form_comp['matches']

England — last 15 competitive matches
  Form score : +0.79
  Record     : 13W 0D 2L



,date,home_team,away_team,home_score,away_score,tournament,elo_contribution
1069,2024-07-14,Spain,England,2,1,UEFA Euro,-1.5961
1070,2024-09-07,Republic of Ireland,England,0,2,UEFA Nations League,0.2779
1071,2024-09-10,England,Finland,2,0,UEFA Nations League,0.2018
1072,2024-10-10,England,Greece,1,2,UEFA Nations League,-1.1932
1073,2024-10-13,Finland,England,1,3,UEFA Nations League,0.2018
1074,2024-11-14,Greece,England,0,3,UEFA Nations League,0.3068
1075,2024-11-17,England,Republic of Ireland,5,0,UEFA Nations League,0.2779
1076,2025-03-21,England,Albania,2,0,FIFA World Cup qualification,0.3762
1077,2025-03-24,England,Latvia,3,0,FIFA World Cup qualification,0.1396
1078,2025-06-07,Andorra,England,0,1,FIFA World Cup qualification,0.0824


In [22]:
# Compare form across several teams
teams = ['Brazil', 'Argentina', 'France', 'England', 'Spain', 'Germany', 'Morocco', 'Japan']

rows = []
for t in teams:
    f_all  = get_form(t, n=10, competitive=False)
    f_comp = get_form(t, n=10, competitive=True)
    rows.append({
        'team':           t,
        'form_all':       f_all['form_score'],
        'form_comp':      f_comp['form_score'],
        'record_all':     f_all['record'],
        'record_comp':    f_comp['record'],
    })

pd.DataFrame(rows).sort_values('form_comp', ascending=False)

,team,form_all,form_comp,record_all,record_comp
6,Morocco,2.8037,3.9552,"(7, 3, 0)","(8, 2, 0)"
3,England,0.6262,2.8973,"(7, 1, 2)","(10, 0, 0)"
4,Spain,0.8872,1.3736,"(7, 3, 0)","(6, 4, 0)"
7,Japan,1.9772,0.4012,"(7, 2, 1)","(7, 2, 1)"
2,France,0.8372,0.1310,"(8, 1, 1)","(7, 1, 2)"
5,Germany,1.6696,-0.1589,"(9, 0, 1)","(6, 1, 3)"
1,Argentina,-0.8608,-1.8034,"(8, 1, 1)","(6, 2, 2)"
0,Brazil,-0.9607,-2.0292,"(6, 1, 3)","(4, 3, 3)"
